# 4. Model Training

## Objective
Train an ensemble of translation models for Akkadian to English translation.

### Approach:
Given the low-resource nature of this task (~1400 training samples), we use:
1. **Fine-tuned MarianMT** - A pre-trained NMT model fine-tuned on our data
2. **T5-base** - Text-to-text transfer transformer
3. **Ensemble** - Combine predictions from multiple models

The evaluation metric is the Geometric Mean of BLEU and chrF++ scores.

### Memory Management:
To avoid GPU out-of-memory errors, we train models one at a time and clear GPU memory after each model.

In [1]:
# Uncomment in Kaggle
#!pip install sacrebleu sacremoses

In [2]:
N_EPOCHS=2
LR=1.e-3
WARMUP=200
BATCH_SIZE=16

In [3]:
# Imports
from pathlib import Path
import pandas as pd
import numpy as np
import json
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import warnings
from transformers import (
    MarianTokenizer, 
    MarianMTModel,
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    T5Tokenizer, 
    T5ForConditionalGeneration
)

from datasets import Dataset as HFDataset
import sacrebleu
from tqdm import tqdm
import random
from transformers import set_seed
print('Libraries Loaded')

Libraries Loaded


Modify the below cells when moving to Kaggle.

In [4]:
OUTPUT_DIR = Path("../output")
MODELS_DIR = OUTPUT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESSED_DIR = OUTPUT_DIR / "pre_processed_data"
ENSEMBLE_DIR = MODELS_DIR / "ensemble"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def clear_gpu_memory():
    """Clear GPU memory to avoid OOM errors when training multiple models."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"GPU memory cleared. Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def get_gpu_memory_info():
    """Get current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        return f"Allocated: {allocated:.2f} GB, Reserved: {reserved:.2f} GB"
    return "No GPU"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("GPU memory management functions defined")
clear_gpu_memory()
get_gpu_memory_info()

Using device: cuda
GPU: NVIDIA GeForce GTX 1650
GPU Memory: 3.90 GB
GPU memory management functions defined
GPU memory cleared. Allocated: 0.00 GB


'Allocated: 0.00 GB, Reserved: 0.00 GB'

## 4.1 Ensemble Config

Train 5 diverse models for ensemble translation:
1. **T5-small (v1)** - Already trained, will be reused
2. **T5-base** - Larger capacity transformer
3. **MarianMT** - Encoder-decoder from Helsinki-NLP
4. **ByT5** - Byte-level model, handles unseen characters
5. **T5-small (v2)** - Different seed for diversity

**IMPORTANT**: Each model is trained separately with GPU memory cleared between models to avoid OOM errors.

The ensemble uses **beam voting** to combine predictions.


In [6]:
# Ensemble Configuration
ENSEMBLE_CONFIG = {
    "models": [
#        {
#            "name": "t5_small_v1",
#            "type": "t5",
#            "model_name": "t5-small",
#            "path": "t5_akkadian_translator",
#            "exists": False  # Already trained
#        },
#        {
#            "name": "t5_base",
#            "type": "t5",
#            "model_name": "t5-base",
#            "path": "ensemble/t5_base",
#            "exists": False
#        },
#        {
#            "name": "marianmt",
#            "type": "marian",
#            "model_name": "Helsinki-NLP/opus-mt-mul-en",
#            "path": "ensemble/marianmt",
#            "exists": False
#        },
#        {
#            "name": "byt5",
#            "type": "t5",
#            "model_name": "google/byt5-small",
#            "path": "ensemble/byt5",
#            "exists": False
#        },
        {
            "name": "t5_small_v2",
            "type": "t5",
            "model_name": "t5-small",
            "path": "ensemble/t5_small_v2",
            "seed": 42,
            "exists": False
        }
    ],
    "beam_size": 5,
    "voting_strategy": "beam_voting"
}

In [7]:
print("Ensemble configuration:")
print(f"  Models: {len(ENSEMBLE_CONFIG['models'])}")
print(f"  Beam size per model: {ENSEMBLE_CONFIG['beam_size']}")
print(f"  Total candidates: {len(ENSEMBLE_CONFIG['models']) * ENSEMBLE_CONFIG['beam_size']}")
print(f"\nEnsemble directory: {ENSEMBLE_DIR}")

Ensemble configuration:
  Models: 1
  Beam size per model: 5
  Total candidates: 5

Ensemble directory: ../output/models/ensemble


## 4.2 Load Preprocessed Data

The data has been preprocessed by notebook 03, which combines:
- Original training data from `data/train.csv`
- Akkadian-English parallel corpus downloaded from Akkademia (ORACC RINAP data)

This combined dataset provides more training examples for the model.

In [8]:
# Load preprocessed data
train_df = pd.read_csv(PREPROCESSED_DIR / "train_split.csv")
val_df = pd.read_csv(PREPROCESSED_DIR / "val_split.csv")

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# Show data source breakdown if available
if 'source' in train_df.columns:
    print(f"\nData sources in training set:")
    print(train_df['source'].value_counts())

Training samples: 37786
Validation samples: 4199

Data sources in training set:
source
akkademia_train    34459
akkademia_valid     1935
train.csv           1392
Name: count, dtype: int64


## 4.3 Create Datasets for Training

In [9]:
# Create HuggingFace datasets
train_dataset = HFDataset.from_pandas(train_df[['transliteration_cleaned', 'translation_cleaned']])
val_dataset = HFDataset.from_pandas(val_df[['transliteration_cleaned', 'translation_cleaned']])

# Rename columns for consistency
train_dataset = train_dataset.rename_column('transliteration_cleaned', 'source')
train_dataset = train_dataset.rename_column('translation_cleaned', 'target')
val_dataset = val_dataset.rename_column('transliteration_cleaned', 'source')
val_dataset = val_dataset.rename_column('translation_cleaned', 'target')

print("Datasets created")
print(train_dataset)

Datasets created
Dataset({
    features: ['source', 'target'],
    num_rows: 37786
})


In [10]:
def preprocess_function(examples):
    """Prepare examples for MarianMT"""
    # Add prefix for source language
    inputs = ["<unk> " + src for src in examples["source"]]
    targets = [tgt for tgt in examples["target"]]
    
    # Tokenize
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    
    # Tokenize targets
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Preprocessing function defined")

Preprocessing function defined


## 4.4 Evaluation Function

In [11]:
# Create evaluation function
def compute_metrics(preds, labels, tokenizer_ref):
    """Compute BLEU and chrF++ metrics"""
    # Decode predictions
    pred_strs = tokenizer_ref.batch_decode(preds, skip_special_tokens=True)
    label_strs = tokenizer_ref.batch_decode(labels, skip_special_tokens=True)
    
    # Filter empty strings
    pred_strs = [p.strip() for p in pred_strs if p.strip()]
    label_strs = [l.strip() for l in label_strs if l.strip()]
    
    # Compute BLEU
    bleu = sacrebleu.corpus_bleu(pred_strs, [label_strs])
    
    # Compute chrF++
    chrf = sacrebleu.corpus_chrf(pred_strs, [label_strs])
    
    # Geometric mean
    geo_mean = np.sqrt(bleu.score * chrf.score)
    
    return {
        "bleu": bleu.score,
        "chrf": chrf.score,
        "geometric_mean": geo_mean
    }

print("Metrics function defined")

Metrics function defined


## 4.4 Training Function Definition

In [12]:
def train_single_model(model_config, train_data, val_data, num_epochs=8, batch_size=16, learning_rate=3e-4):
    """
    Train a single model with GPU memory cleanup.
    Returns the model path and clears GPU memory after saving.
    """
    model_type = model_config["type"]
    model_name = model_config["model_name"]
    model_path = MODELS_DIR / model_config["path"]
    
    # Check if model already exists
    if model_path.exists() and (model_path / "config.json").exists():
        print(f"Model already exists at {model_path}, skipping training.")
        return str(model_path)
    
    # Set seed for reproducibility
    seed = model_config.get("seed", 42)
    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    
    print(f"\n{'='*60}")
    print(f"Training: {model_config['name']}")
    print(f"Model type: {model_type}, Base: {model_name}")
    print(f"{'='*60}")
    print(f"GPU memory before loading: {get_gpu_memory_info()}")
    
    # Clear GPU memory before loading new model
    clear_gpu_memory()
    
    if model_type == "t5":
        tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=False)
        model = T5ForConditionalGeneration.from_pretrained(model_name)
    elif model_type == "marian":
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    print(f"GPU memory after loading: {get_gpu_memory_info()}")
    
    model = model.to(device)
    print(f"GPU memory after moving to device: {get_gpu_memory_info()}")
    
    # Prepare datasets
    def tokenize_for_model(examples):
        inputs = ["translate Akkadian to English: " + src for src in examples["source"]]
        targets = examples["target"]
        model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
        model_inputs["labels"] = np.array(labels["input_ids"])
        return model_inputs
    
    train_tokenized = train_data.map(tokenize_for_model, batched=True, remove_columns=train_data.column_names).with_format("torch")
    val_tokenized = val_data.map(tokenize_for_model, batched=True, remove_columns=val_data.column_names).with_format("torch")
    
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=str(MODELS_DIR / f"{model_config['name']}_training"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=2,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        warmup_steps=WARMUP,
        logging_steps=10,
        save_total_limit=1,
        fp16=False,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        dataloader_num_workers=4,
        logging_dir='../logs',
        report_to="tensorboard",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        data_collator=data_collator,
        processing_class=tokenizer,
    )
    
    # Train
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        print(f"Training for {num_epochs} epochs...")
        trainer.train()
    
    # Save model
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"Model saved to {model_path}")

    # Validate
    print(f'Validating {model_config["name"]}...')
    model.eval()

    all_preds = []
    all_labels = []
    
    for batch in tqdm(DataLoader(val_tokenized, batch_size=8), desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        with torch.no_grad():
            outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128)
        
        all_preds.extend(outputs.cpu().numpy())
        all_labels.extend(labels)
    
    metrics = compute_metrics(all_preds, all_labels, tokenizer)
    print("\n=== Validation Metrics ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.2f}")
    
    # CRITICAL: Clean up GPU memory after saving
    print("Cleaning up GPU memory...")
    del trainer
    del model
    del tokenizer
    del train_tokenized
    del val_tokenized
    clear_gpu_memory()
    
    return str(model_path)

print("Training function defined")

Training function defined


In [13]:
for model_config in ENSEMBLE_CONFIG['models']:
    train_single_model(model_config, train_dataset, val_dataset, num_epochs=N_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LR)


Training: t5_small_v2
Model type: t5, Base: t5-small
GPU memory before loading: Allocated: 0.00 GB, Reserved: 0.00 GB
GPU memory cleared. Allocated: 0.00 GB


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

GPU memory after loading: Allocated: 0.00 GB, Reserved: 0.00 GB
GPU memory after moving to device: Allocated: 0.24 GB, Reserved: 0.25 GB


Map:   0%|          | 0/37786 [00:00<?, ? examples/s]

Map:   0%|          | 0/4199 [00:00<?, ? examples/s]

...

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ../output/models/ensemble/t5_small_v2
Validating t5_small_v2...


Evaluating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 525/525 [05:36<00:00,  1.56it/s]



=== Validation Metrics ===
bleu: 3.11
chrf: 8.73
geometric_mean: 5.21
Cleaning up GPU memory...
GPU memory cleared. Allocated: 0.26 GB


In [14]:
# List all saved models
print("Saved models and files:")
for f in MODELS_DIR.rglob("*"):
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DIR)}: {f.stat().st_size / 1024:.1f} KB")

Saved models and files:
  t5_akkadian_translator/model.safetensors: 236369.0 KB
  t5_akkadian_translator/config.json: 1.5 KB
  t5_akkadian_translator/generation_config.json: 0.1 KB
  t5_akkadian_translator/tokenizer_config.json: 2.3 KB
  t5_akkadian_translator/tokenizer.json: 2367.5 KB
  t5_base_training/runs/Mar18_07-08-18_jimmy-cartop/events.out.tfevents.1773842898.jimmy-cartop.125326.1: 5.1 KB
  t5_small_v2_training/checkpoint-2362/optimizer.pt: 472818.2 KB
  t5_small_v2_training/checkpoint-2362/model.safetensors: 236369.0 KB
  t5_small_v2_training/checkpoint-2362/config.json: 1.5 KB
  t5_small_v2_training/checkpoint-2362/scheduler.pt: 1.4 KB
  t5_small_v2_training/checkpoint-2362/generation_config.json: 0.1 KB
  t5_small_v2_training/checkpoint-2362/rng_state.pth: 14.3 KB
  t5_small_v2_training/checkpoint-2362/tokenizer_config.json: 2.3 KB
  t5_small_v2_training/checkpoint-2362/trainer_state.json: 43.9 KB
  t5_small_v2_training/checkpoint-2362/tokenizer.json: 2367.5 KB
  t5_small_v2

## 4.11 Final GPU Memory Cleanup

In [15]:
# Since we don't have a pre-trained Akkadian->English model, we'll use
# a multilingual approach. We'll create a custom tokenizer for Akkadian.

# For this task, we'll train from scratch with a small transformer
# The key is to use subword tokenization that can handle Akkadian script



# Use a multilingual model that can be fine-tuned
# We'll use opus-mt-mul-en as a base (multi-language to English)
# But first, let's prepare the data format

# Prepare source/target format for MarianMT


In [16]:
clear_gpu_memory()

GPU memory cleared. Allocated: 0.02 GB


In [17]:
# Train ensemble models one at a time
# Uncomment and run cells one at a time to train each model

# NOTE: Due to GPU memory constraints on the GTX 1650 (3.63 GB),
# we recommend training only ONE model at a time and clearing GPU memory.
# The following cells demonstrate the workflow for each model.

print("Ensemble training setup complete.")
print("\nTo train additional models, uncomment and run the cells below one at a time.")
print("Each cell includes GPU memory cleanup to prevent OOM errors.")

Ensemble training setup complete.

To train additional models, uncomment and run the cells below one at a time.
Each cell includes GPU memory cleanup to prevent OOM errors.


In [18]:
# Clear GPU before training any new model
print("Clearing GPU before T5-base training...")
clear_gpu_memory()

# Train T5-base
t5_base_config = ENSEMBLE_CONFIG["models"][1]  # t5_base

# Uncomment to train:
 path_t5_base = train_single_model(
     t5_base_config,
     train_dataset,
     val_dataset,
     num_epochs=N_EPOCHS,
     batch_size=4,  # Reduced batch size for larger model
     learning_rate=LR
 )

IndentationError: unexpected indent (1345344496.py, line 9)

In [ ]:
# Clear GPU before training MarianMT
print("Clearing GPU before MarianMT training...")
clear_gpu_memory()

# Train MarianMT
marianmt_config = ENSEMBLE_CONFIG["models"][2]  # marianmt

# Uncomment to train:
 path_marianmt = train_single_model(
     marianmt_config,
     train_dataset,
     val_dataset,
     num_epochs=N_EPOCHS,
     batch_size=8,
     learning_rate=LR
 )

In [ ]:
# Clear GPU before training ByT5
print("Clearing GPU before ByT5 training...")
clear_gpu_memory()

# Train ByT5
byt5_config = ENSEMBLE_CONFIG["models"][3]  # byt5

# Uncomment to train:
 path_byt5 = train_single_model(
     byt5_config,
     train_dataset,
     val_dataset,
     num_epochs=N_EPOCHS,
     batch_size=4,  # Reduced batch size for ByT5
     learning_rate=LR
 )

In [ ]:
# Clear GPU before training T5-small v2
print("Clearing GPU before T5-small v2 training...")
clear_gpu_memory()

# Train T5-small v2
t5_small_v2_config = ENSEMBLE_CONFIG["models"][4]  # t5_small_v2

# Uncomment to train:
 path_t5_small_v2 = train_single_model(
     t5_small_v2_config,
     train_dataset,
     val_dataset,
     num_epochs=N_EPOCHS,
     batch_size=16,
     learning_rate=LR
 )